# Benchmarking: Read directly from link

This notebooks reads EOPF Zarr and SAFE files directly to benchmark the performance when opening the data. In the original configuration one band of a Sentinel-2 scene is opened. Three experiments are made:
- EOPF Zarr from EODC via http
- SAFE from EODC via http
- SAFE from CDSE via S3

Potential file lists from this [notebook](https://github.com/xcube-dev/xcube-stac/blob/main/examples/notebooks/sentinel_2_cdse.ipynb) and this [notebook](https://eopf-sample-service.github.io/eopf-sample-notebooks/xcube-eopf-sen2/).

eopf:

- https://stac.core.eopf.eodc.eu/collections/sentinel-2-l2a/items/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316
- https://stac.core.eopf.eodc.eu/collections/sentinel-2-l2a/items/S2C_MSIL2A_20250501T104041_N0511_R008_T32UNE_20250501T161558
- https://stac.core.eopf.eodc.eu/collections/sentinel-2-l2a/items/S2B_MSIL2A_20250506T103629_N0511_R008_T32UNE_20250506T115207

cdse equivalents: 

- https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l2a/items/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316
- https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l2a/items/S2C_MSIL2A_20250501T104041_N0511_R008_T32UNE_20250501T161558
- https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l2a/items/S2B_MSIL2A_20250506T103629_N0511_R008_T32UNE_20250506T115207

To view any item in the STAC browser use a a link like this: https://stac.browser.user.eopf.eodc.eu/collections/sentinel-2-l2a/items/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316?.language=de


## Libraries

In [1]:
import xarray as xr
import dask
import time
import psutil
import logging
import pandas as pd

#for direct loading
import rioxarray
import fsspec
import s3fs

## Define paths to files

In [2]:
#path_eopf_zarr = "https://objectstore.eodc.eu:2222/e05ab01a9d56408d82ac32d69a5aae2a:sample-data/tutorial_data/cpm_v253/S2B_MSIL1C_20250113T103309_N0511_R108_T32TLQ_20250113T122458.zarr"
path_eodc_zarr = "https://objectstore.eodc.eu:2222/e05ab01a9d56408d82ac32d69a5aae2a:202505-s02msil2a/03/products/cpm_v256/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316.zarr"
path_eodc_safe = "https://objects.eodc.eu/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/SAFE/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316.SAFE"
path_cdse_safe = "s3://eodata/Sentinel-2/MSI/L2A/2025/05/03/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316.SAFE"

## Set up logger for counting http requests
This sets up the fsspec logger which counts the http requests that are made when accessing the data. Zarr typically has many small requests, whereas SAFE has few large requests.

In [3]:
# Silent in-memory log capture 
fsspec_logs = []

class ListHandler(logging.Handler):
    def __init__(self, storage):
        super().__init__()
        self.storage = storage
    def emit(self, record):
        self.storage.append(self.format(record))

# Disable root logger output to notebook
logging.getLogger().handlers.clear()

# Configure only fsspec.http logger
logger_fsspec = logging.getLogger("fsspec.http")
logger_fsspec.handlers.clear()
logger_fsspec.propagate = False  # <-- important! stops logs bubbling up
logger_fsspec.setLevel(logging.DEBUG)
logger_fsspec.addHandler(ListHandler(fsspec_logs))


This sets up the tracking of CPU and Mem.

In [4]:
# cpu tracking
process = psutil.Process()

This function takes an arbitrary reader function and adds the benchmarking around it.

In [5]:
def benchmark_run(func, repeats=1):
    """
    Benchmark any callable `func` and measure:
      - Wall time
      - CPU time (user + system)
      - Memory delta (MB)
      - HTTP requests (from global `fsspec_logs`)
    Returns a pandas DataFrame with all results.
    """
    process = psutil.Process()
    results = []

    for i in range(repeats):
        fsspec_logs.clear()

        mem_before = process.memory_info().rss
        cpu_before = process.cpu_times()
        t0 = time.perf_counter()

        func()  # run the actual workload

        t1 = time.perf_counter()
        cpu_after = process.cpu_times()
        mem_after = process.memory_info().rss

        results.append({
            "run": i + 1,
            "time_s": t1 - t0,
            "cpu_user_s": cpu_after.user - cpu_before.user,
            "cpu_sys_s": cpu_after.system - cpu_before.system,
            "mem_delta_MB": (mem_after - mem_before) / (1024**2),
            "http_requests": len(fsspec_logs),
        })

    df = pd.DataFrame(results)
    display(df)
    #print("Averages:")
    #display(df.mean(numeric_only=True))
    return df

## Benchmarking

### EOPF Zarr on EODC


Define the reader function for opening the zarr datatree.

In [6]:
def open_zarr():
    xr.open_datatree(path_eodc_zarr, engine="zarr", mask_and_scale=False, chunks={})

Execute benchmarking. 

In [7]:
bm_zarr_datatree = benchmark_run(open_zarr, repeats=3)

,run,time_s,cpu_user_s,cpu_sys_s,mem_delta_MB,http_requests
0,1,1.178004,0.67,0.06,92.980469,59
1,2,0.560937,0.22,0.00,0.773438,59
2,3,0.378448,0.12,0.02,2.050781,59


Assign the data to a variable to check whether the data has been accessed correctly.

In [8]:
dt = xr.open_datatree(path_eodc_zarr, engine="zarr", mask_and_scale=False, chunks={})

Define the reader function to actually load a band of the scene.

In [9]:
def load_band():
    _ = dt["measurements/reflectance/r10m"]["b04"].load()

Execute the benchmarking for loading a band. *Note: After the first run cached results are used.*

In [10]:
bm_zarr_band = benchmark_run(load_band, repeats=3)

,run,time_s,cpu_user_s,cpu_sys_s,mem_delta_MB,http_requests
0,1,0.630904,0.54,0.16,249.382812,36
1,2,0.000460,0.00,0.00,0.000000,0
2,3,0.000288,0.00,0.00,0.000000,0


Check that the data is loaded correctly.

In [11]:
band_eodc_zarr = dt["measurements/reflectance/r10m"]["b04"].load()
band_eodc_zarr

<xarray.DataArray 'b04' (y: 10980, x: 10980)> Size: 241MB
array([[1818, 1915, 1966, ...,    0,    0,    0],
       [1739, 2026, 2208, ...,    0,    0,    0],
       [1690, 2102, 2338, ...,    0,    0,    0],
       ...,
       [3148, 3218, 3404, ...,    0,    0,    0],
       [2872, 3032, 3386, ...,    0,    0,    0],
       [2754, 2932, 3158, ...,    0,    0,    0]], dtype=uint16)
Coordinates:
  * x        (x) int64 88kB 499985 499995 500005 500015 ... 609755 609765 609775
  * y        (y) int64 88kB 5999995 5999985 5999975 ... 5890225 5890215 5890205
Attributes: (12/15)
    _eopf_attrs:     {'add_offset': -0.1, 'coordinates': ['x', 'y'], 'dimensi...
    add_offset:      -0.1
    dtype:           <u2
    fill_value:      0
    long_name:       BOA reflectance from MSI acquisition at spectral band b0...
    proj:bbox:       [499980.0, 5890200.0, 609780.0, 6000000.0]
    ...              ...
    proj:wkt2:       PROJCS["WGS 84 / UTM zone 32N",GEOGCS["WGS 84",DATUM["WG...
    scale_factor:    0.0001
    units:           digital_counts
    valid_max:       65535
    valid_min:       1
    _FillValue:      0

### EOPF SAFE on EODC

Define the path to the band in the SAFE file.

In [12]:
# Full URL to the B04 10m band file, from f"{path_eopf_zarr}/manifest.safe"
b04_url = (
    f"{path_eodc_safe}/GRANULE/L2A_T32UNE_A051514_20250503T103937/IMG_DATA/R10m/"
    "T32UNE_20250503T103701_B04_10m.jp2"
)

Define the function to load a band from the SAFE format.

In [13]:
def load_safe_band():
    fs = fsspec.filesystem("http")
    with fs.open(b04_url, mode="rb") as f:
        _ = rioxarray.open_rasterio(f, masked=False).load()

Execute the benchmarking on loading a SAFE band.

In [14]:
bm_safe_eodc = benchmark_run(load_safe_band, repeats=3)

,run,time_s,cpu_user_s,cpu_sys_s,mem_delta_MB,http_requests
0,1,3.465382,12.32,0.47,842.503906,2
1,2,3.177479,12.12,0.14,545.175781,2
2,3,3.371932,12.85,0.07,344.839844,2


Load the band and check the values.

In [15]:
fs = fsspec.filesystem("http")
with fs.open(b04_url) as f: 
    band_eodc_safe = rioxarray.open_rasterio(f, masked=False).load()

band_eodc_safe

<xarray.DataArray (band: 1, y: 10980, x: 10980)> Size: 241MB
array([[[1818, 1915, 1966, ...,    0,    0,    0],
        [1739, 2026, 2208, ...,    0,    0,    0],
        [1690, 2102, 2338, ...,    0,    0,    0],
        ...,
        [3148, 3218, 3404, ...,    0,    0,    0],
        [2872, 3032, 3386, ...,    0,    0,    0],
        [2754, 2932, 3158, ...,    0,    0,    0]]], dtype=uint16)
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
  * y            (y) float64 88kB 6e+06 6e+06 6e+06 ... 5.89e+06 5.89e+06
    spatial_ref  int64 8B 0
Attributes:
    scale_factor:  1.0
    add_offset:    0.0

### SAFE on CDSE S3

*Note: Counting of http requests is not possible on S3. Another logger would have to be set up to count S3 requests.*

Insert your CDSE S3 credentials. These ones are only valid for a certain amount of time. Here's a [guide](https://documentation.dataspace.copernicus.eu/APIs/S3.html#generate-secrets) how generate them.

In [16]:
# Only valid for a certain amount of time. Check the guide above how to create your own.
credentials = {
    "key": "FTE4ZT820RDZTHOU6I8C",
    "secret": "EdSaK2k1DjJm1rTlbucDaaSsmSSawWFz9da9Wemz",
}

Set up the S3 file system.

In [17]:
fs = s3fs.S3FileSystem(
    key=credentials["key"],
    secret=credentials["secret"],
    client_kwargs={
        "region_name": "eu-central-1",
        "endpoint_url": "https://s3.dataspace.copernicus.eu"
    }
)

Define the path to the band within the SAFE file.

In [18]:
# Correct path from manifest.safe
band_path = (
    "eodata/Sentinel-2/MSI/L2A/2025/05/03/"
    "S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316.SAFE/"
    "GRANULE/L2A_T32UNE_A051514_20250503T103937/IMG_DATA/R10m/"
    "T32UNE_20250503T103701_B04_10m.jp2"
)

Measure timing of loading one band.

In [19]:
%%time
# Open the file from S3
with fs.open(band_path, mode="rb") as f:
    band_cdse_safe = rioxarray.open_rasterio(f, masked=False).load()

CPU times: user 12.4 s, sys: 129 ms, total: 12.6 s
Wall time: 4.51 s


Check the values.

In [20]:
band_cdse_safe

<xarray.DataArray (band: 1, y: 10980, x: 10980)> Size: 241MB
array([[[1818, 1915, 1966, ...,    0,    0,    0],
        [1739, 2026, 2208, ...,    0,    0,    0],
        [1690, 2102, 2338, ...,    0,    0,    0],
        ...,
        [3148, 3218, 3404, ...,    0,    0,    0],
        [2872, 3032, 3386, ...,    0,    0,    0],
        [2754, 2932, 3158, ...,    0,    0,    0]]], dtype=uint16)
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
  * y            (y) float64 88kB 6e+06 6e+06 6e+06 ... 5.89e+06 5.89e+06
    spatial_ref  int64 8B 0
Attributes:
    scale_factor:  1.0
    add_offset:    0.0